In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-21.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder.appName("Spark DataFrames 2").enableHiveSupport().getOrCreate()
print("Driver Python:", sys.executable)
print("Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/03 12:27:46 WARN Utils: Your hostname, Darviks-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.14 instead (on interface en0)
26/06/03 12:27:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/03 12:27:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/03 12:27:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Driver Python: /Users/darvikkunalbanda/DataEngineering/.venv/bin/python
Spark: 4.1.2


In [2]:
##readHiveTable to dataframes
#Method 1

# hive_df = spark.read.table("select * from db_name.table_name")
# hive_df.show()

#read data from mysql database to dataframe

jdbc_config = 'jdbc:mysql://<hostname>:<port>/<database>'

cred = {
    'username' : 'root',
    'pass' : 'cloud'
}

# df = spark.read.jdbc()

df = spark.read.jdbc(url=jdbc_url, properties=creds, table="emp")

df.show()

In [3]:
## Rdd to DataFrame

rdd = spark.sparkContext.parallelize([(1,"Jhon",25),(2,"Jane",30),(3,"Bob",35)])

df = rdd.toDF(["emp_id","emp_name","emp_age"])
df.show()

+------+--------+-------+
|emp_id|emp_name|emp_age|
+------+--------+-------+
|     1|    Jhon|     25|
|     2|    Jane|     30|
|     3|     Bob|     35|
+------+--------+-------+



In [4]:
# Array[dict] to DataFrame

data = [
    {"id": 1, "name": "John", "age": 25}, 
    {"id": 2, "name": "Jane", "age": 30}, 
    {"id": 3, "name": "Bob", "age": 35}
]

df = spark.createDataFrame(data)
df.show()

+---+---+----+
|age| id|name|
+---+---+----+
| 25|  1|John|
| 30|  2|Jane|
| 35|  3| Bob|
+---+---+----+



Transformations in DataFrames

### In Spark SQL

In [5]:
df = spark.read.parquet('/Users/darvikkunalbanda/DataEngineering/DE_Drill/dataset/orders.parquet')
df.show(10,False)

+--------+------------+--------+----------+-------------------+---------+
|order_id|product_name|quantity|unit_price|order_date         |status   |
+--------+------------+--------+----------+-------------------+---------+
|ORD-010 |Doohickey C |3       |29.99     |2025-06-10 12:00:00|delivered|
|ORD-004 |Doohickey C |2       |29.99     |2025-03-12 16:45:00|shipped  |
|ORD-007 |Doohickey C |4       |29.99     |2025-05-02 13:20:00|shipped  |
|ORD-005 |Gadget B    |1       |49.99     |2025-04-01 11:00:00|cancelled|
|ORD-006 |Widget A    |10      |19.99     |2025-04-18 08:30:00|delivered|
|ORD-002 |Gadget B    |1       |49.99     |2025-02-20 14:15:00|delivered|
|ORD-009 |Widget A    |1       |19.99     |2025-06-01 10:00:00|shipped  |
|ORD-003 |Widget A    |5       |19.99     |2025-03-05 09:00:00|pending  |
|ORD-008 |Gadget B    |2       |49.99     |2025-05-25 15:10:00|pending  |
|ORD-001 |Widget A    |3       |19.99     |2025-01-15 10:30:00|shipped  |
+--------+------------+--------+------

In [6]:
df.createOrReplaceTempView("order_table")

In [17]:
spark.sql(""" 
          select product_name , quantity , unit_price from order_table 
          where status = 'shipped' and quantity >= 3
          """).show()

+------------+--------+----------+
|product_name|quantity|unit_price|
+------------+--------+----------+
| Doohickey C|       4|     29.99|
|    Widget A|       3|     19.99|
+------------+--------+----------+



In [19]:
spark.sql("""
          select quantity , count(*)
          from order_table
          group by quantity
          """).show()

+--------+--------+
|quantity|count(1)|
+--------+--------+
|       3|       2|
|       2|       2|
|       4|       1|
|       1|       3|
|      10|       1|
|       5|       1|
+--------+--------+



In [26]:
spark.sql("""
          select * from order_table order by unit_price
          """).show(5)

+--------+------------+--------+----------+-------------------+---------+
|order_id|product_name|quantity|unit_price|         order_date|   status|
+--------+------------+--------+----------+-------------------+---------+
| ORD-003|    Widget A|       5|     19.99|2025-03-05 09:00:00|  pending|
| ORD-006|    Widget A|      10|     19.99|2025-04-18 08:30:00|delivered|
| ORD-001|    Widget A|       3|     19.99|2025-01-15 10:30:00|  shipped|
| ORD-009|    Widget A|       1|     19.99|2025-06-01 10:00:00|  shipped|
| ORD-007| Doohickey C|       4|     29.99|2025-05-02 13:20:00|  shipped|
+--------+------------+--------+----------+-------------------+---------+
only showing top 5 rows


In [34]:
spark.sql("""
          select order_id , count(*) from order_table
          group by order_id
          having count(*) > 1
          """).show(5)

+--------+--------+
|order_id|count(1)|
+--------+--------+
+--------+--------+



### using dataframe

In [12]:
df.select("product_name", "quantity","unit_price")\
    .where("status = 'shipped' ")\
    .show(10,False)

+------------+--------+----------+
|product_name|quantity|unit_price|
+------------+--------+----------+
|Doohickey C |2       |29.99     |
|Doohickey C |4       |29.99     |
|Widget A    |1       |19.99     |
|Widget A    |3       |19.99     |
+------------+--------+----------+



In [31]:
from pyspark.sql.functions import count , col , desc
df.groupBy('unit_price').agg(count("*")).show(5,False)

+----------+--------+
|unit_price|count(1)|
+----------+--------+
|29.99     |3       |
|49.99     |3       |
|19.99     |4       |
+----------+--------+



In [32]:
df.orderBy(desc("unit_price")).show()

+--------+------------+--------+----------+-------------------+---------+
|order_id|product_name|quantity|unit_price|         order_date|   status|
+--------+------------+--------+----------+-------------------+---------+
| ORD-008|    Gadget B|       2|     49.99|2025-05-25 15:10:00|  pending|
| ORD-002|    Gadget B|       1|     49.99|2025-02-20 14:15:00|delivered|
| ORD-005|    Gadget B|       1|     49.99|2025-04-01 11:00:00|cancelled|
| ORD-010| Doohickey C|       3|     29.99|2025-06-10 12:00:00|delivered|
| ORD-004| Doohickey C|       2|     29.99|2025-03-12 16:45:00|  shipped|
| ORD-007| Doohickey C|       4|     29.99|2025-05-02 13:20:00|  shipped|
| ORD-003|    Widget A|       5|     19.99|2025-03-05 09:00:00|  pending|
| ORD-006|    Widget A|      10|     19.99|2025-04-18 08:30:00|delivered|
| ORD-001|    Widget A|       3|     19.99|2025-01-15 10:30:00|  shipped|
| ORD-009|    Widget A|       1|     19.99|2025-06-01 10:00:00|  shipped|
+--------+------------+--------+------

In [ ]:
df.groupBy("order_id").agg(count("*")).where("count(*) > 1").show()

+--------+--------+
|order_id|count(1)|
+--------+--------+
+--------+--------+



26/06/03 13:59:59 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 933706 ms exceeds timeout 120000 ms
26/06/03 13:59:59 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/03 14:00:06 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1530)
	at o